In [ ]:
# ==========================================
# CELL 1: INSTALL DEPENDENCIES
# ==========================================
!pip install roboflow ultralytics torchmetrics pandas matplotlib seaborn opencv-python-headless -q

In [ ]:
# ==========================================
# CELL 2: IMPORT LIBRARIES
# ==========================================
import os
import glob
import zipfile
import shutil
import random
import time
from collections import defaultdict
import numpy as np
import pandas as pd
from PIL import Image
import cv2
import matplotlib.pyplot as plt
import torch
import seaborn as sns
from ultralytics import YOLO
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from roboflow import Roboflow
from IPython.display import display, HTML

print(f"PyTorch version  : {torch.__version__}")
print(f"CUDA available   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU              : {torch.cuda.get_device_name(0)}")

In [ ]:
# ==========================================
# CELL 3: KONFIGURASI
# ==========================================

# Ganti dengan API key dan workspace/project Anda
API_KEY = os.getenv("ROBOFLOW_API_KEY", "qkrwwn0zGtWk0FFezlwt")
WORKSPACE_NAME = "datasetbariq"
PROJECT_NAME   = "violations-in-lab"
VERSION_NUMBER = 1
MODEL_FORMAT   = "yolov8"  # Format untuk YOLOv8

# Device
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Training
NUM_EPOCHS  = 60
BATCH_SIZE  = 8
IMG_SIZE    = 640
PATIENCE    = 10    # Early stopping patience
CONF_THRESHOLD = 0.7
IOU_THRESHOLD  = 0.7

# Kelas (sesuaikan dengan dataset Anda)
CLASS_NAMES = [
    "playing game phone",
    "sleep",
    "eating"
]
NUM_CLASSES = len(CLASS_NAMES)

# Display names untuk laporan skripsi
DISPLAY_NAMES = {
    "playing game phone": "Game (Main HP)",
    "sleep":              "Tidur",
    "eating":             "Makan",
}

print(f"Device  : {DEVICE}")
print(f"Classes : {CLASS_NAMES}")

In [ ]:
# ==========================================
# CELL 4: DOWNLOAD DATASET (HARD FIX)
# ==========================================

# 1. KONFIGURASI LINK
# Paste link dari Tab "Terminal" di Roboflow di sini!
# Contoh format: "https://universe.roboflow.com/ds/xyz...?key=123..."
# DOWNLOAD_LINK = "https://app.roboflow.com/ds/rlCOT4pUUB?key=nUG1xVZEJU" #8020 TANPA PREPROCESSING
# DOWNLOAD_LINK = "https://app.roboflow.com/ds/kV7YQNOe8n?key=IiHDmCrETX" #8020
# DOWNLOAD_LINK = "https://app.roboflow.com/ds/uxbqBCp6sJ?key=UtzrJD8qio" #7030
# DOWNLOAD_LINK = "https://app.roboflow.com/ds/gLz2jeTxRE?key=yQGxelvVLu" #701515
# DOWNLOAD_LINK = "https://app.roboflow.com/ds/3WMToKYpSt?key=dU8plpaHEo" #702010 Augmentation Beda
# DOWNLOAD_LINK = "https://app.roboflow.com/ds/zg7g7jAQAH?key=GjQHfhUtXn" #702010 Tanpa Preprocessing dan Augmentation
# DOWNLOAD_LINK = "https://app.roboflow.com/ds/OsZs5g2Tam?key=AB6Z9peNm7" #702010 Preprocessing AO, RS640
# DOWNLOAD_LINK = "https://app.roboflow.com/ds/UJigMQzLqK?key=gvnPXT41dN" #702010 Preprocessing AO, RS640, AAC Histo
# DOWNLOAD_LINK = "https://app.roboflow.com/ds/skULf30k8d?key=7f5IzBbY9x" #702010 Preprocessing AO, RS640, AAC Histo, Gray
# DOWNLOAD_LINK = "https://app.roboflow.com/ds/szb7Xi7GYj?key=j2VHyVmXqc" #702010 Augmentation Flip, 90r
# DOWNLOAD_LINK = "https://app.roboflow.com/ds/O9vWKp8mEu?key=okzjFVYXfA" #702010 Augmentation Flip, Bright
# DOWNLOAD_LINK = "https://app.roboflow.com/ds/TJrgXBbjHl?key=u1U5eEzZbN" #702010 Augmentation Flip, Shear
# DOWNLOAD_LINK = "https://app.roboflow.com/ds/hwlP8YNwFA?key=lB8LhUQkSY" #702010 Gabungan #1
# DOWNLOAD_LINK = "https://app.roboflow.com/ds/IH53GGhCMp?key=JvFP93WS9G" #702010 Gabungan #2
# New
# DOWNLOAD_LINK = "https://app.roboflow.com/ds/TfK5vmdJYU?key=bdCZ71bspV" #702010 Tanpa Preprocessing dan Augmentation
# DOWNLOAD_LINK = "https://app.roboflow.com/ds/zGFKlQvFUc?key=cJ7v5Cjviq" #702010 Preprocessing AO, RS640
# DOWNLOAD_LINK = "https://app.roboflow.com/ds/zAkjNPZu1x?key=70aVbmGqp1" #702010 Preprocessing AO, RS640, AAC Histo
# DOWNLOAD_LINK = "https://app.roboflow.com/ds/J8bjVYRqH5?key=QjwoCnYHIl" #702010 Preprocessing AO, RS640, AAC Histo, Gray
# DOWNLOAD_LINK = "https://app.roboflow.com/ds/jcO6dXaoFV?key=dlBjpTLm8C" #702010 Preprocessing AO, RFB640
# DOWNLOAD_LINK = "https://app.roboflow.com/ds/McQtJBM8XE?key=OysHUfptlB" #702010 Preprocessing AO, RFB640, AAC Histo
# DOWNLOAD_LINK = "https://app.roboflow.com/ds/3O8DwyVcO0?key=G3ufNcagOj" #702010 Preprocessing AO, RFB640, AAC Histo, Gray
# DOWNLOAD_LINK = "https://app.roboflow.com/ds/64d6Ccjj2z?key=PsafOt9b9y" #702010 Augmentation Flip, 90r
# DOWNLOAD_LINK = "https://app.roboflow.com/ds/O3tF61FgIk?key=2NiOkAGiMU" #702010 Augmentation Flip, Bright
# DOWNLOAD_LINK = "https://app.roboflow.com/ds/pADH4hThi7?key=EQu4BETTF0" #702010 Augmentation Flip, Shear

# DOWNLOAD_LINK = "https://app.roboflow.com/ds/u6MMAK79i2?key=dFc6xGtBjZ" #702010 Gabungan #2
DOWNLOAD_LINK = "https://app.roboflow.com/ds/ou2kQlpAqp?key=Pgr8fYkj7s" #702010 Gabungan #2 RFW

# Folder tujuan yang bersih
DATASET_FOLDER = "/content/dataset_manual"

# 2. BERSIH-BERSIH (Hapus data lama yang rusak)
if os.path.exists(DATASET_FOLDER):
    shutil.rmtree(DATASET_FOLDER)
os.makedirs(DATASET_FOLDER, exist_ok=True)

# 3. DOWNLOAD PAKSA (Menggunakan curl system)
print("⏳ Sedang mendownload dataset...")
zip_path = os.path.join(DATASET_FOLDER, "roboflow.zip")
# -L untuk follow redirect, -o untuk output file
os.system(f'curl -L "{DOWNLOAD_LINK}" -o {zip_path}')

# 4. CEK & EKSTRAK
if os.path.exists(zip_path):
    size = os.path.getsize(zip_path)
    print(f"📦 Ukuran file terdownload: {size/1024:.3f} KB")

    if size < 1000: # Jika kurang dari 1KB, pasti error text
        print("❌ ERROR: File terlalu kecil. Link salah atau API Key bermasalah.")
        with open(zip_path, 'r') as f:
            print("Isi file:", f.read())
    else:
        try:
            print("📂 Sedang mengekstrak...")
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                zip_ref.extractall(DATASET_FOLDER)
            print("✅ Ekstraksi SUKSES!")

            # Hapus zip biar hemat storage
            os.remove(zip_path)

            # Set variabel global agar Cell berikutnya bisa baca
            class DatasetDummy:
                location = DATASET_FOLDER
            dataset = DatasetDummy()

        except zipfile.BadZipFile:
            print("❌ ERROR: File masih corrupt. Coba generate version baru di Roboflow.")
else:
    print("❌ Gagal download. File tidak ditemukan.")

In [ ]:
# ==========================================
# CELL 5: CEK STRUKTUR DATASET & YAML
# ==========================================

if 'dataset' not in globals():
    class DatasetDummy: location = "/content/dataset_manual"
    dataset = DatasetDummy()

DATASET_DIR = dataset.location
print(f"📁 Dataset folder: {DATASET_DIR}")

# Cari file data.yaml
yaml_path = None
for root, dirs, files in os.walk(DATASET_DIR):
    for f in files:
        if f.endswith('.yaml') or f.endswith('.yml'):
            yaml_path = os.path.join(root, f)
            break
    if yaml_path:
        break

if yaml_path:
    print(f"✅ File YAML ditemukan: {yaml_path}")
    with open(yaml_path, 'r') as f:
        print("\n📄 Isi YAML:")
        print(f.read())
else:
    print("❌ File YAML tidak ditemukan. Pastikan dataset format YOLOv8.")

# Cek direktori train/valid/test
for split in ['train', 'valid', 'test']:
    split_dir = os.path.join(DATASET_DIR, split)
    if os.path.exists(split_dir):
        img_count = len(glob.glob(os.path.join(split_dir, 'images', '*')))
        lbl_count = len(glob.glob(os.path.join(split_dir, 'labels', '*')))
        print(f"  {split}: {img_count} gambar, {lbl_count} label")
    else:
        print(f"  {split}: ❌ Tidak ditemukan")

In [ ]:
# ==========================================
# CELL 6: FUNGSI EVALUASI (MIRIP FASTER R-CNN)
# ==========================================

def evaluate_yolo_model(model, data_yaml, split='val', conf_threshold=0.7, iou_threshold=0.7, device='cuda'):
    """
    Evaluasi model YOLOv8 dengan format output mirip Faster R-CNN.
    Menghitung mAP@0.5, mAP@0.5:0.95, Recall, dan FPS.
    """
    model_device = model.device

    # Jalankan evaluasi bawaan YOLOv8
    print("⏳ Menghitung metrik evaluasi...")
    results = model.val(
        data=data_yaml,
        split=split,
        conf=conf_threshold,
        iou=iou_threshold,
        verbose=False,
        device=device
    )

    # Ambil metrik global
    map50_all   = float(results.box.map50)
    map_all     = float(results.box.map)    # mAP 0.5:0.95
    recall_all  = float(results.box.mr)     # mean recall

    # Hitung FPS dari inference speed
    speed       = results.speed  # dict: preprocess, inference, postprocess (ms)
    inf_ms      = speed.get('inference', 0)
    fps         = 1000.0 / inf_ms if inf_ms > 0 else 0

    # Metrik per kelas
    # results.box.ap_class_index -> indeks kelas
    # results.box.ap50           -> AP50 per kelas
    # results.box.ap             -> AP50:95 per kelas
    # results.box.r              -> recall per kelas
    ap50_per_class = results.box.ap50    # ndarray
    ap_per_class   = results.box.ap      # ndarray (AP50:95)
    r_per_class    = results.box.r       # ndarray (recall)
    class_indices  = results.box.ap_class_index  # ndarray

    # Jumlah gambar per split
    total_images = len(results.jdict) if hasattr(results, 'jdict') and results.jdict else 0
    # Fallback: hitung dari names
    if total_images == 0:
        # Estimasi dari results.nt (num targets per kelas)
        if hasattr(results.box, 'nt'):
            total_images = int(results.box.nt.sum()) if hasattr(results.box.nt, 'sum') else 0

    # Nama kelas dari model
    class_names_model = results.names  # dict: id -> name

    # --- CETAK TABEL FORMAT YOLO ---
    print("\n" + "="*85)
    print(f"{'Class':<20} {'Images':<10} {'Recall':<12} {'mAP50':<12} {'mAP50-95':<12}")
    print("-" * 85)

    # Baris 'all'
    print(f"{'all':<20} {total_images:<10} {recall_all:.3f}       {map50_all:.3f}       {map_all:.3f}")

    ap_data = []
    for i, cls_idx in enumerate(class_indices):
        cls_idx = int(cls_idx)
        name    = class_names_model.get(cls_idx, f"class_{cls_idx}")
        r_cls   = float(r_per_class[i])   if i < len(r_per_class)   else 0.0
        ap_cls  = float(ap_per_class[i])  if i < len(ap_per_class)  else 0.0

        print(f"{name:<20} {'':<10} {r_cls:.3f}       {'-':<12} {ap_cls:.3f}")
        ap_data.append([name, r_cls, ap_cls])

    print("="*85 + "\n")

    df_ap = pd.DataFrame(ap_data, columns=['Class', 'Recall', 'mAP50-95'])

    return {
        'mAP@0.5':          map50_all,
        'map_50_95':         map_all,
        'recall':            recall_all,
        'fps':               fps,
        'inference_time_ms': inf_ms,
        'ap_per_class':      df_ap,
    }


def calculate_per_class_metrics_yolo(model, data_yaml, split='test',
                                      conf_threshold=0.7, iou_threshold=0.7,
                                      device='cuda'):
    """
    Menghitung TP, FN, dan Recall PER KELAS untuk YOLOv8.

    Tujuan: Menjawab "kelas mana yang paling sulit dideteksi?"

    Returns:
        DataFrame dengan kolom:
        [Jenis Pelanggaran, Jumlah Data Test (BB), Terdeteksi Benar (TP),
         Gagal Terdeteksi (FN), Akurasi/Recall (%)]
    """
    results = model.val(
        data=data_yaml,
        split=split,
        conf=conf_threshold,
        iou=iou_threshold,
        verbose=False,
        device=device
    )

    class_names_model = results.names  # dict: id -> name
    class_indices     = results.box.ap_class_index  # ndarray

    # nt: number of ground-truth targets per kelas
    # tp: true positives per kelas (dari confusion matrix / stats)
    # YOLOv8 menyimpan stats dalam results.box
    # Gunakan: results.box.tp dan results.box.nt
    nt_per_class = results.box.nt_per_class   # jumlah GT per kelas
    r_per_class  = results.box.r              # recall per kelas

    # Display names untuk laporan skripsi
    display_names = {
        'playing game phone': 'Game (Main HP)',
        'sleep':              'Tidur',
        'eating':             'Makan',
    }

    rows = []
    for i, cls_idx in enumerate(class_indices):
        cls_idx  = int(cls_idx)
        name     = class_names_model.get(cls_idx, f"class_{cls_idx}")
        gt_count = int(nt_per_class[i]) if i < len(nt_per_class) else 0
        recall   = float(r_per_class[i]) if i < len(r_per_class) else 0.0
        tp       = round(recall * gt_count)
        fn       = gt_count - tp
        recall_pct = recall * 100

        rows.append({
            'Jenis Pelanggaran':     display_names.get(name, name),
            'Jumlah Data Test (BB)': gt_count,
            'Terdeteksi Benar (TP)': tp,
            'Gagal Terdeteksi (FN)': fn,
            'Akurasi / Recall (%)':  f"{recall_pct:.1f}%",
        })

    df_result = pd.DataFrame(rows)
    return df_result


print("✅ Fungsi evaluasi siap!")

In [ ]:
# ==========================================
# CELL 7: INISIALISASI MODEL YOLOv8m
# ==========================================

print("🔄 Loading YOLOv8m pretrained weights...")
model = YOLO("yolov8m.pt")   # Auto-download jika belum ada
print(f"✅ Model: YOLOv8m")
print(f"   Parameters : {sum(p.numel() for p in model.model.parameters()):,}")

In [ ]:
# ==========================================
# CELL 8: TRAINING YOLOv8m DENGAN EARLY STOPPING
# ==========================================

start_time = time.time()

print("🚀 Mulai Training dengan Early Stopping...")
print(f"⏸ Patience: {PATIENCE} epoch")

train_results = model.train(
    data=yaml_path,
    epochs=NUM_EPOCHS,
    batch=BATCH_SIZE,
    imgsz=IMG_SIZE,
    device=DEVICE,
    patience=PATIENCE,            # Early stopping otomatis
    save=True,
    save_period=5,                # Simpan checkpoint setiap 5 epoch
    project="/content/yolov8m_runs",
    name="lab_violation",
    exist_ok=True,
    # Augmentasi
    fliplr=0.5,
    flipud=0.0,
    degrees=0.0,
    translate=0.1,
    scale=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    # Optimizer
    optimizer='AdamW',
    lr0=0.0001,
    weight_decay=0.0005,
    warmup_epochs=3,
    close_mosaic=10,
    # Verbose
    verbose=True,
)

total_time = (time.time() - start_time) / 60

# Path ke model terbaik
best_model_path  = "/content/yolov8m_runs/lab_violation/weights/best.pt"
final_model_path = "/content/yolov8m_runs/lab_violation/weights/last.pt"

print("\n" + "★"*60)
print(f"✅ Training Selesai: {total_time:.2f} menit")
print(f"💾 Best model  : {best_model_path}")
print(f"💾 Final model : {final_model_path}")
print("★"*60)

In [ ]:
# ==========================================
# CELL 9: LOAD BEST MODEL & EVALUASI VALIDASI
# ==========================================

print("📦 Loading best model...")
best_model = YOLO(best_model_path)

# Evaluasi pada validation set
print("\n📊 EVALUASI PADA VALIDATION SET:")
val_results = evaluate_yolo_model(
    best_model,
    data_yaml=yaml_path,
    split='val',
    conf_threshold=CONF_THRESHOLD,
    iou_threshold=IOU_THRESHOLD,
    device=DEVICE
)

print(f"📊 HASIL EVALUASI VALIDATION SET:")
print(f"• mAP@0.5        : {val_results['mAP@0.5']:.3f}")
print(f"• mAP@0.5-0.95   : {val_results['map_50_95']:.3f}")
print(f"• Recall         : {val_results['recall']:.3f}")
print(f"• FPS            : {val_results['fps']:.1f}")
print(f"• Inference Time : {val_results['inference_time_ms']:.1f} ms")
print("-" * 40)
if 'ap_per_class' in val_results:
    print("Per Kelas:")
    print(val_results['ap_per_class'].to_string(index=False))
print("="*60)

In [ ]:
# ==========================================
# CELL 10: VISUALISASI GRAFIK TRAINING
# ==========================================

import glob as _glob

def plot_yolo_training_metrics(run_dir="/content/yolov8m_runs/lab_violation"):
    """
    Membaca results.csv dari YOLOv8 training dan memplot grafik
    dengan layout 2x3 mirip Faster R-CNN.
    """
    csv_path = os.path.join(run_dir, 'results.csv')
    if not os.path.exists(csv_path):
        print(f"❌ File results.csv tidak ditemukan di {run_dir}")
        return

    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()  # Hapus whitespace
    print(f"📄 Kolom tersedia: {list(df.columns)}")

    epochs = df['epoch'].values if 'epoch' in df.columns else range(1, len(df)+1)
    num_epochs = len(epochs)

    # --- SETUP LAYOUT 2 BARIS x 3 KOLOM ---
    fig, axs = plt.subplots(2, 3, figsize=(20, 12))
    fig.suptitle(f'Evaluasi Model YOLOv8m (Total {num_epochs} Epochs)', fontsize=18, y=0.98)

    def plot_line(ax, x, y_col, title, color_train, color_val=None, label_train='Train', label_val='Val'):
        # Kolom bisa berbeda tergantung versi ultralytics
        possible_train = [y_col, f'train/{y_col}', f'train_{y_col}']
        possible_val   = [f'val/{y_col}', f'val_{y_col}', f'metrics/{y_col}']

        train_col = next((c for c in possible_train if c in df.columns), None)
        val_col   = next((c for c in possible_val   if c in df.columns), None)

        if train_col:
            ax.plot(x, df[train_col].values, color=color_train, label=label_train, linewidth=2)
        if val_col and color_val:
            ax.plot(x, df[val_col].values, color=color_val, label=label_val, marker='o', linewidth=2)

        ax.set_title(title, fontweight='bold')
        ax.set_xlabel('Epoch')
        ax.grid(True, linestyle='--', alpha=0.6)
        ax.legend()

    # --- BARIS 1 ---
    # 1. Total Box Loss (Train vs Val)
    ax = axs[0, 0]
    train_box = next((c for c in ['train/box_loss', 'train/box_om', 'box_loss'] if c in df.columns), None)
    val_box   = next((c for c in ['val/box_loss',   'val/box_om',   'val_box_loss'] if c in df.columns), None)
    if train_box:
        ax.plot(epochs, df[train_box].values, 'r-', label='Training Box Loss', linewidth=2)
    if val_box:
        ax.plot(epochs, df[val_box].values, 'g-o', label='Validation Box Loss', linewidth=2)
    ax.set_title('Box Loss: Training vs Validation', fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.grid(True, linestyle='--')
    ax.legend()

    # 2. mAP@0.5
    ax = axs[0, 1]
    map50_col = next((c for c in ['metrics/mAP50(B)', 'metrics/mAP_0.5', 'mAP50'] if c in df.columns), None)
    if map50_col:
        ax.plot(epochs, df[map50_col].values, 'b-o', label='mAP@0.5', linewidth=2)
    ax.set_title('Validation mAP@0.5', fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.grid(True, linestyle='--', alpha=0.6)
    ax.legend()

    # 3. Class Loss (Train vs Val)
    ax = axs[0, 2]
    train_cls = next((c for c in ['train/cls_loss', 'cls_loss'] if c in df.columns), None)
    val_cls   = next((c for c in ['val/cls_loss',   'val_cls_loss'] if c in df.columns), None)
    if train_cls:
        ax.plot(epochs, df[train_cls].values, 'orange', label='Train Cls Loss', linewidth=2)
    if val_cls:
        ax.plot(epochs, df[val_cls].values, 'darkorange', label='Val Cls Loss', marker='o', linewidth=2)
    ax.set_title('Classification Loss', fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.grid(True, linestyle='--', alpha=0.6)
    ax.legend()

    # --- BARIS 2 ---
    # 4. DFL Loss
    ax = axs[1, 0]
    train_dfl = next((c for c in ['train/dfl_loss', 'dfl_loss'] if c in df.columns), None)
    val_dfl   = next((c for c in ['val/dfl_loss',   'val_dfl_loss'] if c in df.columns), None)
    if train_dfl:
        ax.plot(epochs, df[train_dfl].values, 'orange', label='Train DFL Loss', linewidth=2)
    if val_dfl:
        ax.plot(epochs, df[val_dfl].values, 'darkorange', label='Val DFL Loss', marker='o', linewidth=2)
    ax.set_title('DFL (Distribution Focal) Loss', fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.grid(True, linestyle='--', alpha=0.6)
    ax.legend()

    # 5. Precision
    ax = axs[1, 1]
    prec_col = next((c for c in ['metrics/precision(B)', 'metrics/precision', 'precision'] if c in df.columns), None)
    if prec_col:
        ax.plot(epochs, df[prec_col].values, 'green', label='Precision', linewidth=2)
    ax.set_title('Validation Precision', fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.grid(True, linestyle='--', alpha=0.6)
    ax.legend()

    # 6. Recall
    ax = axs[1, 2]
    rec_col = next((c for c in ['metrics/recall(B)', 'metrics/recall', 'recall'] if c in df.columns), None)
    if rec_col:
        ax.plot(epochs, df[rec_col].values, 'green', label='Recall', linewidth=2)
    ax.set_title('Validation Recall', fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.grid(True, linestyle='--', alpha=0.6)
    ax.legend()

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

plot_yolo_training_metrics()

In [ ]:
# ==========================================
# CELL 11: EVALUASI TEST SET
# ==========================================

# Cek apakah ada folder test
test_dir = os.path.join(DATASET_DIR, 'test')
if not os.path.exists(test_dir):
    print("⚠️  Folder 'test' tidak ditemukan, menggunakan 'val' sebagai test set")
    test_split = 'val'
else:
    test_split = 'test'
    test_img_count = len(glob.glob(os.path.join(test_dir, 'images', '*')))
    print(f"✅ Menggunakan test set: {test_img_count} gambar")

print("\n📊 EVALUASI PADA TEST SET:")
test_results = evaluate_yolo_model(
    best_model,
    data_yaml=yaml_path,
    split=test_split,
    conf_threshold=CONF_THRESHOLD,
    iou_threshold=IOU_THRESHOLD,
    device=DEVICE
)

print(f"\n📊 HASIL AKHIR TEST SET:")
print(f"• mAP@0.5        : {test_results['mAP@0.5']:.3f}")
print(f"• mAP@0.5-0.95   : {test_results['map_50_95']:.3f}")
print(f"• Recall         : {test_results['recall']:.3f}")
print(f"• FPS            : {test_results['fps']:.1f}")
print(f"• Inference Time : {test_results['inference_time_ms']:.1f} ms")
print("-" * 40)
if 'ap_per_class' in test_results:
    print("Per Kelas:")
    print(test_results['ap_per_class'].to_string(index=False))
print("="*60)

In [ ]:
# ==========================================
# CELL 12: TABEL PER KELAS (FORMAT SKRIPSI)
# ==========================================

print("\n📋 MENGHITUNG METRIK PER KELAS (FORMAT SKRIPSI)...")

df_per_class = calculate_per_class_metrics_yolo(
    best_model,
    data_yaml=yaml_path,
    split=test_split,
    conf_threshold=CONF_THRESHOLD,
    iou_threshold=IOU_THRESHOLD,
    device=DEVICE
)

display(df_per_class)

# Analisis otomatis
if len(df_per_class) > 0:
    recall_vals = df_per_class['Akurasi / Recall (%)'].str.replace('%','').astype(float)
    best_class  = df_per_class.loc[recall_vals.idxmax(), 'Jenis Pelanggaran']
    worst_class = df_per_class.loc[recall_vals.idxmin(), 'Jenis Pelanggaran']
    print(f"\n📊 Analisis Otomatis:")
    print(f"  Paling mudah: {best_class} (Recall {recall_vals.max():.1f}%)")
    print(f"  Paling sulit: {worst_class} (Recall {recall_vals.min():.1f}%)")

In [ ]:
# ==========================================
# CELL 13: VISUALISASI DETEKSI + TABEL DATA
# ==========================================

def visualize_test_with_table(model, test_images_dir, conf_threshold=0.7, max_images=None):
    """
    Menampilkan visualisasi deteksi dan tabel hasil per gambar.
    Mirip dengan fungsi di Faster R-CNN notebook.
    """
    # Cari semua gambar test
    img_paths = sorted(glob.glob(os.path.join(test_images_dir, 'images', '*')))
    if not img_paths:
        # Fallback: cari langsung di folder
        img_paths = sorted(glob.glob(os.path.join(test_images_dir, '*.jpg')) +
                           glob.glob(os.path.join(test_images_dir, '*.jpeg')) +
                           glob.glob(os.path.join(test_images_dir, '*.png')))

    if max_images:
        img_paths = img_paths[:max_images]

    total_imgs = len(img_paths)
    print(f"📂 Menampilkan {total_imgs} gambar dari Data Test (Urut)...")

    # Nama kelas dari model
    class_names_model = model.names  # dict: id -> name

    # Warna per kelas (BGR untuk OpenCV)
    colors_bgr = [
        (0,   0,   255),  # Merah
        (255, 0,   0  ),  # Biru
        (0,   255, 0  ),  # Hijau
        (255, 255, 0  ),  # Cyan
        (0,   255, 255),  # Kuning
    ]

    for idx, img_path in enumerate(img_paths):
        fname = os.path.basename(img_path)

        # Prediksi
        preds = model.predict(
            img_path,
            conf=conf_threshold,
            verbose=False
        )[0]

        # Load gambar
        img_np = cv2.imread(img_path)
        img_np = cv2.cvtColor(img_np, cv2.COLOR_BGR2RGB)

        boxes  = preds.boxes.xyxy.cpu().numpy()    # (N, 4)
        scores = preds.boxes.conf.cpu().numpy()    # (N,)
        labels = preds.boxes.cls.cpu().numpy().astype(int)  # (N,)

        # List untuk tabel deteksi
        detections_data = []

        img_draw = img_np.copy()
        for i in range(len(boxes)):
            box        = boxes[i].astype(int)
            label_id   = labels[i]
            score      = scores[i]
            label_name = class_names_model.get(label_id, f"class_{label_id}")
            color_rgb  = colors_bgr[label_id % len(colors_bgr)][::-1]  # BGR -> RGB
            color_bgr  = colors_bgr[label_id % len(colors_bgr)]

            # Simpan data untuk tabel
            detections_data.append({
                "No":          i + 1,
                "Pelanggaran": label_name,
                "Confidence":  f"{score:.3f}"
            })

            # Gambar bbox di img_draw (RGB)
            img_draw_bgr = cv2.cvtColor(img_draw, cv2.COLOR_RGB2BGR)
            cv2.rectangle(img_draw_bgr, (box[0], box[1]), (box[2], box[3]), color_bgr, 2)
            text = f"{label_name}: {score:.3f}"
            (w, h), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 1)
            cv2.rectangle(img_draw_bgr, (box[0], box[1] - 20), (box[0] + w, box[1]), color_bgr, -1)
            cv2.putText(img_draw_bgr, text, (box[0], box[1] - 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
            img_draw = cv2.cvtColor(img_draw_bgr, cv2.COLOR_BGR2RGB)

        # --- TAMPILKAN HASIL ---
        plt.figure(figsize=(10, 6))
        plt.imshow(img_draw)
        plt.axis('off')
        plt.title(f"{fname} (Threshold: {conf_threshold})")
        plt.show()

        if len(detections_data) > 0:
            df = pd.DataFrame(detections_data)
            print(f"📋 Hasil Deteksi untuk {fname}:")
            display(df)
        else:
            print(f"⚪ Tidak ada pelanggaran terdeteksi pada {fname} (Score < {conf_threshold})")

        print("-" * 80)

# Jalankan Visualisasi
# max_images=None untuk tampilkan semua, atau set angka tertentu misal max_images=10
visualize_test_with_table(
    best_model,
    test_images_dir=os.path.join(DATASET_DIR, test_split),
    conf_threshold=CONF_THRESHOLD,
    max_images=None
)

In [ ]:
# ==========================================
# CELL 14: SIMPAN MODEL & EKSPOR KE ONNX
# ==========================================

# Salin best model ke lokasi yang mudah diakses
shutil.copy(best_model_path, '/content/best_yolov8m.pt')
print(f"✅ Best model disimpan: /content/best_yolov8m.pt")

# (Opsional) Ekspor ke ONNX untuk deployment
# best_model.export(format='onnx', imgsz=IMG_SIZE)
# print("✅ Model diekspor ke ONNX")

print("\n📊 RINGKASAN AKHIR:")
print("="*60)
print(f"Model            : YOLOv8m")
print(f"Dataset          : {DATASET_DIR}")
print(f"Conf Threshold   : {CONF_THRESHOLD}")
print(f"IoU Threshold    : {IOU_THRESHOLD}")
print("-"*60)
print(f"mAP@0.5          : {test_results['mAP@0.5']:.3f}")
print(f"mAP@0.5-0.95     : {test_results['map_50_95']:.3f}")
print(f"Recall           : {test_results['recall']:.3f}")
print(f"FPS              : {test_results['fps']:.1f}")
print(f"Inference Time   : {test_results['inference_time_ms']:.1f} ms")
print("="*60)